# 02. Preprocessing & Evaluation

전처리, 분류기 평가 파이프라인, 결과 시각화.

**사전 조건**: `01_Setup_and_Data_Loading.ipynb` 실행 완료 (또는 아래 셀에서 직접 로딩)

## 0. 데이터 로딩 (독립 실행용)
이미 01번을 실행했다면 이 셀을 건너뛰세요.

In [ ]:
# 01번 노트북의 전체 코드를 실행합니다 (독립 실행용)
# %run ./01_Setup_and_Data_Loading.ipynb

# 또는 아래 주석 해제 후 직접 실행
# exec(open('01_setup_code.py').read())

## 1. 전처리 유틸리티

In [ ]:
def _flatten_dict_cell(cell):
    if isinstance(cell, dict):
        flat = []
        for key in sorted(cell.keys()):
            val = cell[key]
            if isinstance(val, np.ndarray): flat.extend(val.flatten().tolist())
            elif isinstance(val, (list, tuple)): flat.extend(list(val))
            else: flat.append(float(val))
        return flat
    elif isinstance(cell, np.ndarray): return cell.flatten().tolist()
    elif isinstance(cell, (list, tuple)): return list(cell)
    return [float(cell)]

def preprocess_X(X):
    """X를 float64 ndarray로 정규화 (object dtype 지원)."""
    if hasattr(X, 'numpy'):  X = X.numpy()
    if hasattr(X, 'values'): X = X.values
    if not isinstance(X, np.ndarray): X = np.array(X)
    if X.dtype == object:
        rows = []
        for row in X:
            flat = []
            if hasattr(row, '__iter__') and not isinstance(row, (str, dict)):
                for c in row: flat.extend(_flatten_dict_cell(c))
            else: flat.extend(_flatten_dict_cell(row))
            rows.append(flat)
        max_len = max(len(r) for r in rows)
        rows = [r + [0.0]*(max_len - len(r)) for r in rows]
        X = np.array(rows, dtype=np.float64)
    X = X.astype(np.float64)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

def preprocess_y(y):
    if hasattr(y, 'numpy'):  y = y.numpy()
    if hasattr(y, 'values'): y = y.values
    return np.array(y).flatten().astype(int)

print('Preprocessing functions defined.')

## 2. 분류기 평가 함수

In [ ]:
def get_all_classifiers():
    clfs = {
        'KNN (k=3)':     KNeighborsClassifier(3),
        'KNN (k=12)':    KNeighborsClassifier(12),
        'SVM (RBF)':     SVC(kernel='rbf', C=1., gamma='scale'),
        'SVM (Linear)':  SVC(kernel='linear', C=1.),
        'Random Forest': RandomForestClassifier(100, random_state=RANDOM_STATE),
    }
    for C in C_VALUES:
        clfs[f'Soft-SVM (C={C})'] = SVC(kernel='rbf', C=C, gamma='scale')
    return clfs

def evaluate_all_classifiers(X, y, n_splits=N_SPLITS):
    """모든 분류기에 대해 Soft Acc, Strict Acc, Macro F1 계산."""
    clfs = get_all_classifiers()
    results = {}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    for name, ct in clfs.items():
        acc_soft, acc_strict, f1_macro = [], [], []
        all_y_true, all_y_pred = [], []
        for tri, tei in skf.split(X, y):
            c = clone(ct); c.fit(X[tri], y[tri]); yp = c.predict(X[tei])
            acc_soft.append(soft_accuracy_score(y[tei], yp))
            acc_strict.append(accuracy_score(y[tei], yp))
            f1_macro.append(f1_score(y[tei], yp, average='macro', zero_division=0))
            all_y_true.extend(y[tei]); all_y_pred.extend(yp)
        results[name] = {
            'mean_soft': np.mean(acc_soft)*100, 'std_soft': np.std(acc_soft)*100,
            'mean_strict': np.mean(acc_strict)*100, 'std_strict': np.std(acc_strict)*100,
            'mean_f1': np.mean(f1_macro)*100, 'std_f1': np.std(f1_macro)*100,
            'y_true': np.array(all_y_true), 'y_pred': np.array(all_y_pred),
        }
    return results

def full_evaluate(X, y, reduction_dim=REDUCTION_DIM):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    if X_scaled.shape[1] > reduction_dim:
        X_reduced = PCA(n_components=reduction_dim, random_state=RANDOM_STATE).fit_transform(X_scaled)
    else:
        X_reduced = X_scaled
    clf_results = evaluate_all_classifiers(X_reduced, y)
    return {'classifiers': clf_results, 'original_dim': X.shape[1],
            'reduced_dim': X_reduced.shape[1], 'X_reduced': X_reduced, 'y': y}

CLASSIFIERS_LIST = ['KNN (k=3)','KNN (k=12)','SVM (RBF)','SVM (Linear)',
                    'Random Forest','Soft-SVM (C=0.5)','Soft-SVM (C=1.0)','Soft-SVM (C=2.0)']
print(f'Evaluation functions defined: PCA={REDUCTION_DIM}D, {N_SPLITS}-fold CV')

## 3. 결과 테이블 & 시각화

In [ ]:
def print_comparison_table(all_results, use_soft=True):
    metric = 'soft' if use_soft else 'strict'
    title  = 'Adjacent Tolerance' if use_soft else 'Strict'
    print(f"\n{'='*130}")
    print(f'Full Classifier Comparison ({title})')
    print(f"{'='*130}")
    header = f"{'Method':<20} {'Dim':>8}"
    for c in CLASSIFIERS_LIST: header += f' {c[:14]:>16}'
    print(header); print('-'*130)
    for method, result in all_results.items():
        row = f"{method[:20]:<20} {result.get('original_dim','N/A'):>8}"
        cr = result.get('classifiers', {})
        for c in CLASSIFIERS_LIST:
            if c in cr:
                m = cr[c][f'mean_{metric}']; s = cr[c][f'std_{metric}']
                row += f' {m:>7.1f}+/-{s:<5.1f}%'
            else: row += f" {'N/A':>16}"
        print(row)
    print('='*130)

def plot_soft_svm_bar(all_results, C_values=C_VALUES, use_soft=True, save_path=None):
    metric = 'soft' if use_soft else 'strict'
    methods = list(all_results.keys())
    x = np.arange(len(methods)); width = 0.25
    fig, ax = plt.subplots(figsize=(max(12, len(methods)*1.5), 6))
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    for i, C in enumerate(C_values):
        means, stds = [], []
        for m in methods:
            key = f'Soft-SVM (C={C})'
            cr = all_results[m].get('classifiers', {})
            means.append(cr.get(key, {}).get(f'mean_{metric}', 0))
            stds.append(cr.get(key, {}).get(f'std_{metric}', 0))
        bars = ax.bar(x+i*width, means, width, yerr=stds, label=f'C={C}',
                      color=colors[i], capsize=3, alpha=0.8)
        for b, mv in zip(bars, means):
            if mv > 0: ax.text(b.get_x()+b.get_width()/2, b.get_height()+2,
                               f'{mv:.1f}', ha='center', va='bottom', fontsize=8, rotation=90)
    suffix = '(Adjacent Tolerance)' if use_soft else '(Strict)'
    ax.set_xlabel('Vectorization Method'); ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'Soft Margin SVM Classification Accuracy {suffix}')
    ax.set_xticks(x + width); ax.set_xticklabels(methods, rotation=45, ha='right')
    ax.legend(title='C'); ax.set_ylim([0, 110]); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

## 4. 실행 — 전체 평가

In [ ]:
print('\n' + '=' * 80)
print(f'EVALUATION  (PCA {REDUCTION_DIM}D, {N_SPLITS}-fold CV)')
print('=' * 80)

all_results = {}
for method_name in METHODS:
    data = datasets[method_name]
    print(f'\n--- [{method_name}] ---')
    X = preprocess_X(data['X'])
    y = preprocess_y(data['y'])
    print(f'  Shape: {X.shape}')
    t0 = time.time()
    res = full_evaluate(X, y)
    all_results[method_name] = res
    cr = res['classifiers']
    best_clf = max(cr, key=lambda k: cr[k]['mean_soft'])
    r = cr[best_clf]
    print(f"  Best: {best_clf} -> Soft={r['mean_soft']:.2f}% | "
          f"Strict={r['mean_strict']:.2f}% | F1={r['mean_f1']:.2f}%  ({time.time()-t0:.1f}s)")

# 결과 출력
if all_results:
    print_comparison_table(all_results, use_soft=True)
    print_comparison_table(all_results, use_soft=False)
    plot_soft_svm_bar(all_results, use_soft=True,
                      save_path=os.path.join(OUTPUT_DIR, 'soft_svm_adjacent_tol.png'))
    plot_soft_svm_bar(all_results, use_soft=False,
                      save_path=os.path.join(OUTPUT_DIR, 'soft_svm_strict.png'))

print('\nEVALUATION COMPLETE')

## 5. 종합 결과 테이블 (Strict + F1)

In [ ]:
for method in METHODS:
    print(f'\n--- {method} (dim={all_results[method]["original_dim"]}) ---')
    cr = all_results[method]['classifiers']
    print(f"{'Classifier':<22} {'Soft(%)':>14} {'Strict(%)':>14} {'F1(%)':>14}")
    print('-' * 64)
    for clf_name in CLASSIFIERS_LIST:
        if clf_name in cr:
            r = cr[clf_name]
            print(f"{clf_name:<22} {r['mean_soft']:.2f}±{r['std_soft']:.2f}  "
                  f"{r['mean_strict']:.2f}±{r['std_strict']:.2f}  "
                  f"{r['mean_f1']:.2f}±{r['std_f1']:.2f}")